# Ejercicio 05

Crea una clase llamada `Ubicacion` que administre la latitud y longitud de un punto en la Tierra. Deberás sobrecargar los operadores necesarios para poder sumar un desplazamiento, comparar si dos ubicaciones son idénticas e imprimir la coordenada de forma entendible.

Requisitos de la clase:
1. `__init__(self, latitud, longitud)`: el constructor debe validar que la latitud esté entre -90 y 90, y la longitud entre -180 y 180.
2. `__str__(self)`: debe devolver el formato legible: `(latitud° N/S, longitud° E/O)`.
3. `__add__(self, otro)`: debe permitir sumar un desplazamiento y devolver una nueva ubicación.
4. `__sub__(self, otro)`: debe permitir restar un desplazamiento y devolver un desplazamiento en términos de latitud y longitud.
5. `__eq__(self, otro)`: debe devolver `True` si ambas coordenadas son iguales (redondeando con `round()`).


In [1]:
import json
import math
import time
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

class Ubicacion:
    def __init__(self, latitud, longitud):
        if not (-90 <= latitud <= 90):
            raise ValueError("La latitud debe estar entre -90 y 90.")
        if not (-180 <= longitud <= 180):
            raise ValueError("La longitud debe estar entre -180 y 180.")

        self.latitud = latitud
        self.longitud = longitud

    def __str__(self):
        latitud = abs(self.latitud)
        longitud = abs(self.longitud)
        lat_dir = 'N' if self.latitud >= 0 else 'S'
        lon_dir = 'E' if self.longitud >= 0 else 'O'
        return f"({latitud:.1f}° {lat_dir}, {longitud:.1f}° {lon_dir})"

    def __add__(self, otro):
        if not isinstance(otro, Ubicacion):
            return NotImplemented
        return Ubicacion(self.latitud + otro.latitud, self.longitud + otro.longitud)

    def __sub__(self, otro):
        if not isinstance(otro, Ubicacion):
            return NotImplemented
        return Ubicacion(self.latitud - otro.latitud, self.longitud - otro.longitud)

    def __eq__(self, otro):
        if not isinstance(otro, Ubicacion):
            return NotImplemented
        return (round(self.latitud, 2), round(self.longitud, 2)) == (round(otro.latitud, 2), round(otro.longitud, 2))

    def obtener_direccion(self):
        parametros = urlencode({
            "lat": self.latitud,
            "lon": self.longitud,
            "format": "jsonv2",
            "accept-language": "es",
        })
        url = f"https://nominatim.openstreetmap.org/reverse?{parametros}"
        solicitud = Request(
            url,
            headers={"User-Agent": "EjercicioUbicacion/1.0 (uso educativo)"},
        )

        try:
            with urlopen(solicitud, timeout=10) as respuesta:
                resultado = json.load(respuesta)
        except HTTPError as error:
            raise RuntimeError(f"Error HTTP de Nominatim: {error.code}.") from error
        except URLError as error:
            raise RuntimeError(f"No se pudo conectar con Nominatim: {error.reason}.") from error

        direccion = resultado.get("display_name")
        if direccion:
            return direccion

        raise LookupError("Nominatim no encontró una dirección para estas coordenadas.")

    def distancia_a(self, otro):
        if not isinstance(otro, Ubicacion):
            raise TypeError("La distancia solo se puede calcular entre dos ubicaciones.")

        lat1 = math.radians(self.latitud)
        lat2 = math.radians(otro.latitud)
        delta_lat = math.radians(otro.latitud - self.latitud)
        delta_lon = math.radians(otro.longitud - self.longitud)
        a = (
            math.sin(delta_lat / 2) ** 2
            + math.cos(lat1) * math.cos(lat2) * math.sin(delta_lon / 2) ** 2
        )
        a = min(1.0, max(0.0, a))
        angulo = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
        return 6371.0088 * angulo

    def __repr__(self):
        return self.__str__()


## ¿Qué hace cada método especial?

Ejecuta el ejemplo para ver el resultado de cada sobrecarga. `__init__` comprueba los límites y guarda los valores (no devuelve un resultado visible); `__str__` convierte el objeto en texto legible; `__add__` crea una ubicación nueva al sumar; `__sub__` calcula la diferencia de coordenadas; y `__eq__` compara las coordenadas redondeadas a dos decimales. También se muestra `__repr__`, la representación del objeto usada en depuración.

Si una coordenada está fuera de los límites, el constructor lanza `ValueError`; el ejemplo lo captura para mostrar el mensaje sin detener el resto de la celda.

El método `distancia_a(otro)` calcula además la distancia geodésica aproximada entre dos puntos sobre la superficie terrestre mediante la fórmula de Haversine; devuelve kilómetros usando un radio medio de la Tierra de 6.371,0088 km. No es la distancia por carretera.

In [2]:
# Cada bloque muestra qué resultado produce el método especial.
#
p1 = Ubicacion(41.63, -4.76)
p2 = Ubicacion(10.00, -5.00)
print(f"1) __init__: p1 guarda latitud={p1.latitud} y longitud={p1.longitud}")

try:
    Ubicacion(91, 0)
except ValueError as error:
    print(f"   Validación: {error}")

print(f"2) __str__: p1 se muestra como {p1}")
print(f"   __repr__: {p1!r}")

suma = p1 + p2
print(f"3) __add__: ({p1.latitud}, {p1.longitud}) + ({p2.latitud}, {p2.longitud}) = {suma}")

diferencia = p1 - p2
print(f"4) __sub__: ({p1.latitud}, {p1.longitud}) - ({p2.latitud}, {p2.longitud}) = diferencia ({diferencia.latitud}°, {diferencia.longitud}°)")

misma_coordenada = Ubicacion(41.39, 2.17)
coordenada_distinta = Ubicacion(41.40, 2.17)
coordenada_redondeada = Ubicacion(41.391, 2.171)
print(f"5) __eq__: p1 == misma_coordenada -> {p1 == misma_coordenada}")
print(f"   __eq__: p1 == coordenada_distinta -> {p1 == coordenada_distinta}")
print(f"   __eq__: compara tras redondear a 2 decimales -> {p1 == coordenada_redondeada}")

# La consulta a Nominatim se realiza en la celda indicada más abajo.
print("Dirección: ejecuta la celda Nominatim para consultar la ubicación aproximada.")


1) __init__: p1 guarda latitud=41.63 y longitud=-4.76
   Validación: La latitud debe estar entre -90 y 90.
2) __str__: p1 se muestra como (41.6° N, 4.8° O)
   __repr__: (41.6° N, 4.8° O)
3) __add__: (41.63, -4.76) + (10.0, -5.0) = (51.6° N, 9.8° O)
4) __sub__: (41.63, -4.76) - (10.0, -5.0) = diferencia (31.630000000000003°, 0.2400000000000002°)
5) __eq__: p1 == misma_coordenada -> False
   __eq__: p1 == coordenada_distinta -> False
   __eq__: compara tras redondear a 2 decimales -> False
Dirección: ejecuta la celda Nominatim para consultar la ubicación aproximada.


## Cambio de API: Google Maps a Nominatim (OpenStreetMap)

Se sustituyó Google Maps Geocoding por **Nominatim**, el servicio de geocodificación de OpenStreetMap. Ya no se necesita API key ni configurar facturación. El método `obtener_direccion()` realiza una petición GET a `https://nominatim.openstreetmap.org/reverse` con los parámetros `lat`, `lon`, `format=jsonv2` y `accept-language=es`, e identifica la aplicación mediante un `User-Agent`. De la respuesta JSON utiliza el campo `display_name`.

### Límites y atribución

El servicio público tiene capacidad limitada: no superar **1 petición por segundo**, identificar la aplicación con un `User-Agent` propio y hacer consultas bajo demanda. Muestra la atribución **© OpenStreetMap contributors** con enlace a https://www.openstreetmap.org/copyright. La dirección es el objeto adecuado más cercano del mapa y puede no coincidir exactamente con el punto. Los datos se distribuyen bajo licencia ODbL. Para un uso frecuente o de producción, consulta la [política oficial de uso](https://operations.osmfoundation.org/policies/nominatim/) y considera otro proveedor o alojar una instancia propia.

In [3]:
# Consultamos las direcciones de ambos puntos respetando el límite de Nominatim.
print("Coordenadas p1:", p1)
print("Dirección aproximada de p1:", p1.obtener_direccion())

time.sleep(1.1)

print("Coordenadas p2:", p2)
print("Dirección aproximada de p2:", p2.obtener_direccion())

distancia_km = p1.distancia_a(p2)
print(f"Distancia en línea geodésica entre p1 y p2: {distancia_km:,.2f} km")
print("Datos de mapa: © OpenStreetMap contributors — https://www.openstreetmap.org/copyright")

Coordenadas p1: (41.6° N, 4.8° O)
Dirección aproximada de p1: 1, Calle de Manuel Jiménez Alfaro, Parquesol, Valladolid, Castilla y León, 47016, España
Coordenadas p2: (10.0° N, 5.0° O)
Dirección aproximada de p2: Tchologo, Savanes, Costa de Marfil
Distancia en línea geodésica entre p1 y p2: 3,517.18 km
Datos de mapa: © OpenStreetMap contributors — https://www.openstreetmap.org/copyright
